In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from collections import Counter
from dataclasses import asdict

import matplotlib.pyplot as plt
import torch
import transformers
from datasets import load_from_disk

from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa import (
    FULL_UNIQUE_PROMPT_COUNTS,
    AccuracyRunConfig,
    deduplicate,
    load_accuracy_results,
    normalize_rows,
    run_accuracy,
    summarize_paper_random,
    summarize_token_lengths,
    summarize_unique_prompts,
    summarize_verdicts,
)
from jlens_reasoning.benchmarks.flenqa.dataset import TASKS
from jlens_reasoning.evaluation import GenerationStatus, ModelOutput

OUTPUT_DIR = context.runs_dir / "flenqa-accuracy"
LENGTHS = (250, 500, 1000, 2000, 3000)

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["train"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
prompts = deduplicate(rows)
assert len(rows) == 12_000
assert len(prompts) == 9_862
{"source_rows": len(rows), "unique_prompts": len(prompts)}

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()

In [ ]:
def generate_output(prompt: str, *, max_new_tokens: int) -> ModelOutput:
    encoded = tokenizer(prompt, return_tensors="pt", truncation=False)
    input_ids = encoded["input_ids"].to(context.device)
    attention_mask = encoded["attention_mask"].to(context.device)
    with torch.inference_mode():
        generated = causal_lm.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            max_new_tokens=max_new_tokens,
        )
    generated_ids = generated[0, input_ids.shape[1] :].tolist()
    eos_ids = causal_lm.generation_config.eos_token_id
    eos_token_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or ())
    complete = bool(generated_ids and generated_ids[-1] in eos_token_ids)
    scored_ids = generated_ids[:-1] if complete else generated_ids
    return ModelOutput(
        text=tokenizer.decode(scored_ids, skip_special_tokens=True),
        token_ids=tuple(generated_ids),
        token_pieces=tuple(
            tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
            for token_id in generated_ids
        ),
        generation_status=(
            GenerationStatus.COMPLETE if complete else GenerationStatus.TRUNCATED
        ),
        finish_reason="eos" if complete else "length",
    )

In [ ]:
manifest = run_accuracy(
    rows,
    output_dir=OUTPUT_DIR,
    tokenizer=tokenizer,
    generate=generate_output,
    config=AccuracyRunConfig(
        model_name=MODEL_NAME,
        tokenizer_name=MODEL_NAME,
        code_revision=PROJECT_COMMIT,
        expected_source_rows=12_000,
        expected_prompts=9_862,
    ),
)
manifest

In [ ]:
results = load_accuracy_results(OUTPUT_DIR, manifest)
assert results.num_rows == 9_862
actual_counts = Counter(int(value) for value in results.column("ctx_size").to_pylist())
assert dict(actual_counts) == FULL_UNIQUE_PROMPT_COUNTS
{"rows": results.num_rows, "counts_by_length": dict(actual_counts)}

In [ ]:
paper_points = summarize_paper_random(results)
assert tuple(point.ctx_size for point in paper_points) == LENGTHS
assert all(point.total == 600 for point in paper_points)
display([asdict(point) | {"accuracy": point.accuracy} for point in paper_points])
plt.figure(figsize=(8, 4.5))
plt.plot(
    [point.ctx_size for point in paper_points],
    [point.accuracy for point in paper_points],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — paper weighting")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
unique_points = summarize_unique_prompts(results)
assert {point.ctx_size: point.total for point in unique_points} == (
    FULL_UNIQUE_PROMPT_COUNTS
)
display([asdict(point) | {"accuracy": point.accuracy} for point in unique_points])
plt.figure(figsize=(8, 4.5))
plt.plot(
    [point.ctx_size for point in unique_points],
    [point.accuracy for point in unique_points],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — unique prompts")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.5))
for task in sorted(TASKS):
    points = summarize_unique_prompts(results, task=task)
    plt.plot(
        [point.ctx_size for point in points],
        [point.accuracy for point in points],
        marker="o",
        label=task,
    )
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA unique-prompt accuracy by task")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
verdict_counts = summarize_verdicts(results)
token_lengths = summarize_token_lengths(results)
display("Verdict counts by nominal length", [asdict(point) for point in verdict_counts])
display(
    "Exact Qwen token lengths by nominal bucket",
    [asdict(point) for point in token_lengths],
)